In [1]:
import pandas as pd
from pathlib import Path

# ==============================
# CONFIG
# ==============================

BASE_DIR = Path("NBA CSV's")

POS_TO_COL = {
    "PG": "pg_percent",
    "SG": "sg_percent",
    "SF": "sf_percent",
    "PF": "pf_percent",
    "C":  "c_percent",
}

# Stats we'll project
STAT_COLS = {
    "pts":  ("pts_per_game",  "pts"),
    "ast":  ("ast_per_game",  "ast"),
    "trb":  ("trb_per_game",  "trb"),
}

# ==============================
# UTILS
# ==============================

def clean_columns(df):
    df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")
    return df


def clean_player_name(name):
    if not isinstance(name, str):
        return ""
    return (
        name.lower()
        .replace("ö", "o")
        .replace("ó", "o")
        .replace(".", "")
        .replace(",", "")
        .strip()
    )


def add_clean_names(df):
    df["player_clean"] = df["player"].apply(clean_player_name)
    return df


# ==============================
# LOAD + MERGE
# ==============================

def load_data():
    pg = clean_columns(pd.read_csv(BASE_DIR / "Player Per Game.csv"))
    per36 = clean_columns(pd.read_csv(BASE_DIR / "Per 36 Minutes.csv"))
    per100 = clean_columns(pd.read_csv(BASE_DIR / "Per 100 Poss.csv"))
    totals = clean_columns(pd.read_csv(BASE_DIR / "Player Totals.csv"))
    pbp = clean_columns(pd.read_csv(BASE_DIR / "Player Play By Play.csv"))
    teams = clean_columns(pd.read_csv(BASE_DIR / "Team Abbrev.csv"))
    return pg, per36, per100, totals, pbp, teams


def merge_all():
    pg, per36, per100, totals, pbp, teams = load_data()

    # Team map
    team_map = dict(zip(teams["team"], teams["abbreviation"]))
    for df in [pg, per36, per100, totals, pbp]:
        df["team"] = df["team"].map(team_map).fillna(df["team"])

    # Ensure season is int
    for df in [pg, per36, per100, totals, pbp]:
        df["season"] = df["season"].astype(int)

    # Merge
    df = pg.merge(per36, on=["player_id", "season", "team"], how="left", suffixes=("", "_per36"))
    df = df.merge(per100, on=["player_id", "season", "team"], how="left", suffixes=("", "_per100"))
    df = df.merge(totals, on=["player_id", "season", "team"], how="left", suffixes=("", "_totals"))
    df = df.merge(pbp, on=["player_id", "season", "team"], how="left", suffixes=("", "_pbp"))

    # Add cleaned names
    df = add_clean_names(df)

    print("Merged dataset shape:", df.shape)
    return df


# ==============================
# FIND INJURED PLAYER
# ==============================

def get_injured_row(df, name):
    name_clean = clean_player_name(name)
    subset = df[df["player_clean"] == name_clean]

    if subset.empty:
        # try last name suggestion
        last = name.split()[-1]
        sug = df[df["player"].str.contains(last, case=False, na=False)]["player"].unique()[:10]
        raise ValueError(f"Player '{name}' not found. Suggestions: {sug}")

    # in case of multiple teams/seasons (trades)
    return subset.sort_values("season", ascending=False).iloc[0]


# ==============================
# MAIN PREDICTION LOGIC
# ==============================

def predict_minutes_after_injury(df, injured_name):
    injured = get_injured_row(df, injured_name)

    team = injured["team"]
    season = injured["season"]
    pos = str(injured["pos"]).split("-")[0]  # handle "SG-SF" etc

    pos_col = POS_TO_COL[pos]

    # Minutes freed
    freed = injured["mp_per_game"]

    # Teammates at same position
    teammates = df[
        (df["team"] == team) &
        (df["season"] == season) &
        (df[pos_col] > 0) &
        (df["player_clean"] != injured["player_clean"])
    ].copy()

    teammates["minute_share"] = teammates[pos_col] / 100
    teammates["predicted_new_minutes"] = teammates["mp_per_game"] + freed * teammates["minute_share"]

    # Project stats
    for stat, (pg_col, totals_col) in STAT_COLS.items():
        if pg_col in teammates.columns:
            # per-minute = per-game / current minutes
            per_min = teammates[pg_col] / teammates["mp_per_game"].replace(0, 1)
            teammates[f"pred_{stat}"] = per_min * teammates["predicted_new_minutes"]

    return teammates.sort_values("predicted_new_minutes", ascending=False).reset_index(drop=True)


# ==============================
# RUN EXAMPLE
# ==============================
if __name__ == "__main__":
    df = merge_all()

    injured_name = "Dennis Schroder"  # example

    try:
        result = predict_minutes_after_injury(df, injured_name)
        print(result.head(10))
    except Exception as e:
        print("Error:", e)


Merged dataset shape: (33034, 146)
   season   lg             player  player_id   age team pos  g   gs  \
0    2026  NBA       Devin Carter  cartede02  23.0  SAC  PG  3  0.0   
1    2026  NBA  Russell Westbrook  westbru01  37.0  SAC  PG  5  2.0   
2    2026  NBA         Malik Monk   monkma01  27.0  SAC  SG  5  0.0   

   mp_per_game  ...  offensive_foul_drawn  points_generated_by_assists  and1  \
0          8.0  ...                   0.0                          6.0   0.0   
1         24.4  ...                   0.0                         46.0   1.0   
2         25.2  ...                   4.0                         26.0   0.0   

   fga_blocked       player_clean  minute_share  predicted_new_minutes  \
0          1.0       devin carter          1.00                 39.600   
1          6.0  russell westbrook          0.45                 38.620   
2          2.0         malik monk          0.03                 26.148   

    pred_pts  pred_ast  pred_trb  
0   6.435000  3.465000  4.9

In [2]:
df = merge_all()
predict_minutes_after_injury(df, "VJ Edgecombe")


Merged dataset shape: (33034, 146)


,season,lg,player,player_id,age,team,pos,g,gs,mp_per_game,...,offensive_foul_drawn,points_generated_by_assists,and1,fga_blocked,player_clean,minute_share,predicted_new_minutes,pred_pts,pred_ast,pred_trb
0,2026,NBA,Quentin Grimes,grimequ01,25.0,PHI,SG,5,0.0,32.0,...,2.0,49.0,1.0,4.0,quentin grimes,0.42,48.884,26.580675,5.49945,6.72155
1,2026,NBA,Eric Gordon,gordoer01,37.0,PHI,SG,2,0.0,8.0,...,0.0,0.0,0.0,0.0,eric gordon,0.70,36.140,18.070000,0.00000,0.00000
2,2026,NBA,Hunter Sallis,sallihu01,22.0,PHI,SG,1,0.0,1.0,...,0.0,0.0,0.0,0.0,hunter sallis,0.39,16.678,0.000000,0.00000,0.00000


In [3]:
# ================================================
#  MACHINE LEARNING: TRAIN STAT PREDICTION MODELS
# ================================================
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Features the model will use (all per-game style)
ML_FEATURE_COLS = [
    "predicted_new_minutes",   # during training this will be mp_per_game
    "fga_per_game",
    "x3pa_per_game",
    "ft_per_game",
    "ast_per_game",
    "tov_per_game",
    "orb_per_game",
    "drb_per_game",
    "trb_per_game",
]


def build_training_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a training dataset from your merged dataframe.
    We train on normal season stats, but use mp_per_game
    as 'predicted_new_minutes' so at prediction time we can
    plug in the real predicted_new_minutes from your injury model.
    """
    train_df = df.copy()

    # Only players with some real run
    if "g" in train_df.columns:
        train_df = train_df[train_df["g"] >= 10]
    train_df = train_df[train_df["mp_per_game"] > 0]

    # This is what the model will see as minutes feature
    train_df["predicted_new_minutes"] = train_df["mp_per_game"]

    needed_cols = ML_FEATURE_COLS + ["pts_per_game", "ast_per_game", "trb_per_game"]
    train_df = train_df.dropna(subset=needed_cols)

    missing = [c for c in needed_cols if c not in train_df.columns]
    if missing:
        raise KeyError(f"Missing columns for ML training: {missing}")

    return train_df


def train_ml_models(df: pd.DataFrame):
    """
    Train 3 RandomForest models to predict season
    pts_per_game, ast_per_game, trb_per_game.
    """
    train_df = build_training_table(df)

    X = train_df[ML_FEATURE_COLS]
    y_pts = train_df["pts_per_game"]
    y_ast = train_df["ast_per_game"]
    y_trb = train_df["trb_per_game"]

    X_train, X_val, y_pts_train, y_pts_val = train_test_split(
        X, y_pts, test_size=0.2, random_state=42
    )

    # Points model
    rf_pts = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf_pts.fit(X_train, y_pts_train)
    pts_pred = rf_pts.predict(X_val)
    print("MAE (points):", mean_absolute_error(y_pts_val, pts_pred))

    # Assists model – reuse same split indices
    _, _, y_ast_train, y_ast_val = train_test_split(
        X, y_ast, test_size=0.2, random_state=42
    )
    rf_ast = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf_ast.fit(X_train, y_ast_train)
    ast_pred = rf_ast.predict(X_val)
    print("MAE (assists):", mean_absolute_error(y_ast_val, ast_pred))

    # Rebounds model
    _, _, y_trb_train, y_trb_val = train_test_split(
        X, y_trb, test_size=0.2, random_state=42
    )
    rf_trb = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf_trb.fit(X_train, y_trb_train)
    trb_pred = rf_trb.predict(X_val)
    print("MAE (rebounds):", mean_absolute_error(y_trb_val, trb_pred))

    models = {"pts": rf_pts, "ast": rf_ast, "trb": rf_trb}
    return models, ML_FEATURE_COLS


def predict_with_ml(
    df: pd.DataFrame,
    injured_player_name: str,
    models,
    feature_cols: list[str],
) -> pd.DataFrame:
    """
    1. Use your existing predict_minutes_after_injury() to get
       replacement players + predicted_new_minutes.
    2. Feed those rows into the trained ML models to get
       ml_pts, ml_ast, ml_trb.
    """
    base = predict_minutes_after_injury(df, injured_player_name).copy()

    missing = [c for c in feature_cols if c not in base.columns]
    if missing:
        raise KeyError(f"Missing ML feature columns in candidates: {missing}")

    X_new = base[feature_cols]

    base["ml_pts"] = models["pts"].predict(X_new)
    base["ml_ast"] = models["ast"].predict(X_new)
    base["ml_trb"] = models["trb"].predict(X_new)

    return base


In [4]:
# =====================================
#  OPPONENT / MATCHUP ADJUSTMENT LAYER
# =====================================

def _find_first_col_with(df: pd.DataFrame, substrings):
    """Utility: find first column containing any of the given substrings."""
    cols = list(df.columns)
    for sub in substrings:
        for c in cols:
            if sub in c:
                return c
    raise KeyError(f"Could not find any column containing {substrings} in {cols}")

def load_opp_table() -> pd.DataFrame:
    """
    Load Opponent Stats Per Game + map team names to abbreviations.
    Returns a df with columns like:
      season, team (abbrev), opp_pts..., opp_ast..., opp_trb...
    """
    opp = clean_columns(pd.read_csv(BASE_DIR / "Opponent Stats Per Game.csv"))
    teams = clean_columns(pd.read_csv(BASE_DIR / "Team Abbrev.csv"))

    # Map full team names to abbreviations
    if "team" not in teams.columns or "abbreviation" not in teams.columns:
        raise KeyError("Team Abbrev.csv must have 'team' and 'abbreviation' columns.")
    team_map = dict(zip(teams["team"], teams["abbreviation"]))

    opp["team"] = opp["team"].map(team_map).fillna(opp["team"])
    if "season" in opp.columns:
        opp["season"] = opp["season"].astype(int)

    return opp

def adjust_for_matchup(ml_df: pd.DataFrame, opponent_abbrev: str, season: int, opp_df: pd.DataFrame):
    """
    Take the ML predictions and adjust them based on how many
    points / assists / rebounds the opponent typically allows.

    Very simple logic:
      factor_pts = opp_pts_allowed / league_avg_opp_pts_allowed
    final_pts = ml_pts * factor_pts
    (same for ast, trb)
    """
    # Identify relevant columns in opp_df
    pts_col = _find_first_col_with(opp_df, ["opp_pts"])
    ast_col = _find_first_col_with(opp_df, ["opp_ast"])
    trb_col = _find_first_col_with(opp_df, ["opp_trb"])

    # Filter to that season
    season_opp = opp_df[opp_df["season"] == season].copy()
    if season_opp.empty:
        raise ValueError(f"No opponent stats found for season {season}.")

    # League averages for that season
    league_pts = season_opp[pts_col].mean()
    league_ast = season_opp[ast_col].mean()
    league_trb = season_opp[trb_col].mean()

    # Opponent row
    row = season_opp[season_opp["team"] == opponent_abbrev]
    if row.empty:
        raise ValueError(f"No opponent row for team {opponent_abbrev} in season {season}.")
    row = row.iloc[0]

    factor_pts = row[pts_col] / league_pts if league_pts else 1.0
    factor_ast = row[ast_col] / league_ast if league_ast else 1.0
    factor_trb = row[trb_col] / league_trb if league_trb else 1.0

    out = ml_df.copy()
    # If opponent allows more than average → factors > 1 → we scale up
    out["final_pts"] = out["ml_pts"] * factor_pts
    out["final_ast"] = out["ml_ast"] * factor_ast
    out["final_trb"] = out["ml_trb"] * factor_trb

    return out

def predict_full_with_matchup(
    df: pd.DataFrame,
    injured_name: str,
    opponent_abbrev: str,
    models,
    feature_cols,
    opp_df: pd.DataFrame,
):
    """
    Full pipeline:
      1. minutes redistribution (your existing logic)
      2. ML pts/ast/trb prediction
      3. matchup adjustment using opponent stats

    Returns a dataframe with:
      predicted_new_minutes, ml_pts/ml_ast/ml_trb, final_pts/final_ast/final_trb
    """
    # Step 1 + 2: minutes + ML
    ml_base = predict_with_ml(df, injured_name, models, feature_cols)

    # Assume all rows are same season as injured player
    season = int(ml_base["season"].iloc[0])

    # Step 3: matchup adjustment
    full = adjust_for_matchup(ml_base, opponent_abbrev, season, opp_df)
    return full


In [5]:
# 1. Build master dataframe (your existing merge function)
df = merge_all()

# 2. Train ML models on historical stats
models, feature_cols = train_ml_models(df)

# 3. Load opponent defensive table
opp_df = load_opp_table()

# 4. Run final predictions for Schroder out vs Suns (PHO)
result = predict_full_with_matchup(
    df,
    injured_name="Dennis Schroder",
    opponent_abbrev="PHO",   # Suns abbreviation from Team Abbrev.csv
    models=models,
    feature_cols=feature_cols,
    opp_df=opp_df,
)

# 5. Look at key numbers
result[
    [
        "player",
        "predicted_new_minutes",
        "ml_pts", "ml_ast", "ml_trb",
        "final_pts", "final_ast", "final_trb",
    ]
]


Merged dataset shape: (33034, 146)
MAE (points): 0.5441174036157701
MAE (assists): 0.00017425397517337713
MAE (rebounds): 0.00019788717056036674


,player,predicted_new_minutes,ml_pts,ml_ast,ml_trb,final_pts,final_ast,final_trb
0,Devin Carter,39.600,1.881,0.7,1.0,1.937509,0.755627,1.054747
1,Russell Westbrook,38.620,12.127,3.8,4.8,12.491316,4.101977,5.062787
2,Malik Monk,26.148,12.242,2.4,1.2,12.609771,2.590723,1.265697


In [6]:
def force_team_abbrev(df, player_name, correct_abbrev):
    """
    Forcefully sets the team abbreviation for all rows
    matching the given player's name.
    """
    name_clean = clean_player_name(player_name)
    mask = df["player_clean"] == name_clean
    
    if mask.sum() == 0:
        print(f"Warning: Player '{player_name}' not found in DF.")
        return df
    
    df.loc[mask, "team"] = correct_abbrev
    print(f"✔ Forced team for {player_name}: {correct_abbrev}")
    return df

# Example: Giannis should be on MIL
df = force_team_abbrev(df, "Giannis Antetokounmpo", "MIL")


✔ Forced team for Giannis Antetokounmpo: MIL


In [7]:
result = predict_full_with_matchup(
    df,
    injured_name="Giannis Antetokounmpo",
    opponent_abbrev="MIA",   # The team MIL is versing
    models=models,
    feature_cols=feature_cols,
    opp_df=opp_df,
)

result[[
    "player",
    "predicted_new_minutes",
    "ml_pts", "ml_ast", "ml_trb",
    "final_pts", "final_ast", "final_trb"
]]


,player,predicted_new_minutes,ml_pts,ml_ast,ml_trb,final_pts,final_ast,final_trb
0,Kyle Kuzma,51.552,9.0070,2.0,5.00,8.740866,1.958463,5.194004
1,Thanasis Antetokounmpo,34.800,0.9225,0.0,0.02,0.895242,0.000000,0.020776
2,Bobby Portis,33.304,10.9235,0.8,5.80,10.600738,0.783385,6.025044
3,Taurean Prince,26.408,5.6535,1.4,1.40,5.486453,1.370924,1.454321
4,Andre Jackson Jr.,7.108,0.2995,0.0,0.02,0.290651,0.000000,0.020776


In [8]:
from model import run_prediction

# Example: Dennis Schroder out vs Suns, and we care about Monk's lines
lines = {
    "pts": 14.5,   # market points line you choose
    "ast": 4.5,    # assists line
    "trb": 3.5,    # rebounds line
}

df = run_prediction("VJ Edgecombe", "BRK", lines)

df[[
    "player",
    "predicted_new_minutes",
    "final_pts", "prob_pts_over",
    "final_ast", "prob_ast_over",
    "final_trb", "prob_trb_over",
]].head(5)


Merged dataset shape: (33034, 82)
RMSE per-minute (pts): 0.03800
RMSE per-minute (ast): 0.00251
RMSE per-minute (trb): 0.00822


TypeError: cannot convert the series to <class 'float'>

In [ ]:
opp_df["team"].unique()


array(['ATL', 'BOS', 'BRK', 'CHI', 'CHH', 'CLE', 'DAL', 'DNN', 'DET',
       'GSW', 'HOU', 'INA', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN',
       'NOP', 'NYK', 'OKC', 'ORL', 'PHI', 'PHO', 'POR', 'SAC', 'SAA',
       'TOR', 'UTA', 'WAS', 'League Average', 'CHA', 'NOH', 'NJN', 'SEA',
       'NOK', 'VAN', 'WSB', 'KCK', 'SDC', 'NOJ', 'BUF', 'NYA', 'KEN',
       'SDS', 'SSL', 'UTS', 'VIR', 'MMS', 'SDA', 'KCO', 'CAR', 'DNR',
       'MMT', 'CAP', 'DLC', 'BLB', 'FLO', 'MMP', 'PTC', 'CIN', 'TEX',
       'SDR', 'SFW', 'LAS', 'MMF', 'NOB', 'PTP', 'WSC', 'HSM', 'MNP',
       'OAK', 'ANA', 'MNM', 'NJA', 'STL', 'CHZ', 'SYR', 'CHP', 'PHW',
       'MNL', 'FTW', 'ROC', 'MLH', 'INO', 'TRI', 'AND', 'CHS', 'SHE',
       'STB', 'WAT', 'INJ', 'PRO', 'CLR', 'DTF', 'PIT', 'TRH'],
      dtype=object)